In [ ]:
import pickle as pkl
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import os
import random
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

# Import protein class from notebook
class Protein:
    def __init__(self, atoms=[], aas=None, cas=None):
        self.atoms = atoms
        self.contact_maps = []
        self.contact_maps_config = []

        if len(atoms) > 0:
            self.get_Ca()
            self.get_D()
        elif aas is not None and cas is not None:
            self.aa = aas
            self.ca = cas
            self.len = len(self.aa)
            self.get_D()
        else:
            print('Data missing.')

    def get_oh(self, aas):
        idx = np.array([aas.index(aa) for aa in self.aa])
        oh = np.eye(len(aas))[idx]
        return oh

    def get_Ca(self):
        coords, aa = [], []
        for atom in self.atoms:
            if atom[3] == 'CA':
                coords.append(list(atom[-1]))
                aa.append(atom[2])
        self.aa = aa
        self.ca = coords
        self.len = len(self.aa)

    def get_D(self):
        self.D = cdist(self.ca, self.ca, metric='euclidean')

    def add_contact_map(self, threshold_lower=0., threshold_upper=12., save=True):
        contact = self.D.copy()
        contact[self.D<threshold_lower] = 0.
        contact[self.D>threshold_upper] = 0.
        contact[(self.D>=threshold_lower) & (self.D<=threshold_upper)] = 1.
        if save:
            self.contact_maps.append(contact)
            self.contact_maps_config.append({'lower': threshold_lower, 'upper': threshold_upper})
        else:
            return contact

    def get_padded_D(self, max_len):
        arr = np.zeros((max_len, max_len))
        arr[:self.len,:self.len] = self.D
        return arr

    def get_padded_aa(self, max_len, aas):
        idx = np.array([aas.index(aa) for aa in self.aa])
        oneHot = np.eye(len(aas))[idx]
        padded_oneHot = np.zeros((max_len, len(aas)))
        padded_oneHot[:self.len] = oneHot
        return padded_oneHot

    def explode(self, length):
        new_aas = []
        new_cas = []
        for i in range(length):
            new_aas.append(self.aa[-length+i:] + self.aa[:i])
            new_cas.append(self.ca[-length+i:] + self.ca[:i])
        for i in range(length, self.len):
            new_aas.append(self.aa[i-length:i])
            new_cas.append(self.ca[i-length:i])
        return (new_aas, new_cas)

class Database:
    def __init__(self):
        self.proteins = []

    def __getitem__(self, idx):
        if isinstance(idx, slice):
            new_db = Database()
            new_db.proteins = self.proteins[idx]
            return new_db
        return self.proteins[idx]

    def append(self, data):
        self.proteins.append(data)

    def extend(self, data):
        self.proteins.extend(data)

    @property
    def len(self):
        return len(self.proteins)

    @property
    def randomize(self):
        random.shuffle(self.proteins)


# Graph Attention Layer for GRAN
class GraphAttention(nn.Module):
    def __init__(self, in_features, out_features, n_heads=8, dropout=0.1, alpha=0.2):
        super(GraphAttention, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.n_heads = n_heads
        self.dropout = dropout
        self.alpha = alpha

        # Define trainable parameters
        self.W = nn.Parameter(torch.zeros(size=(in_features, n_heads * out_features)))
        self.a = nn.Parameter(torch.zeros(size=(2 * out_features, 1)))

        # Initialize parameters
        nn.init.xavier_normal_(self.W.data)
        nn.init.xavier_normal_(self.a.data)

        # Define layers
        self.dropout_layer = nn.Dropout(self.dropout)
        self.leakyrelu = nn.LeakyReLU(self.alpha)

    def forward(self, h, adj):
        # h: node features [batch_size, N, in_features]
        # adj: adjacency matrix [batch_size, N, N]

        batch_size, N = h.size(0), h.size(1)

        # Linear transformation
        Wh = torch.matmul(h, self.W) # [batch_size, N, n_heads * out_features]
        Wh = Wh.view(batch_size, N, self.n_heads, self.out_features) # [batch_size, N, n_heads, out_features]

        # Compute attention coefficients
        Wh_repeated_in_chunks = Wh.repeat_interleave(N, dim=1) # [batch_size, N * N, n_heads, out_features]
        Wh_repeated_alternating = Wh.repeat(1, N, 1, 1) # [batch_size, N * N, n_heads, out_features]

        # Concatenate to get all possible node pairs
        all_combinations = torch.cat([Wh_repeated_in_chunks, Wh_repeated_alternating], dim=3) # [batch_size, N * N, n_heads, 2*out_features]

        # Reshape for computing attention coefficients
        all_combinations = all_combinations.view(batch_size, N, N, self.n_heads, 2 * self.out_features)

        # Compute attention coefficients
        e = self.leakyrelu(torch.matmul(all_combinations, self.a).squeeze(-1)) # [batch_size, N, N, n_heads]

        # Mask attention coefficients using adjacency matrix
        adj = adj.unsqueeze(3).expand(batch_size, N, N, self.n_heads)
        e = e.masked_fill(adj == 0, float('-inf'))

        # Apply softmax to get attention weights
        attention = F.softmax(e, dim=2) # [batch_size, N, N, n_heads]
        attention = self.dropout_layer(attention)

        # Apply attention weights to node features
        h_prime = torch.zeros(batch_size, N, self.n_heads, self.out_features).to(h.device)

        for b in range(batch_size):
            for i in range(N):
                for j in range(N):
                    if adj[b, i, j, 0] == 1:
                        h_prime[b, i] += attention[b, i, j].unsqueeze(-1) * Wh[b, j]

        # Reshape output
        h_prime = h_prime.view(batch_size, N, self.n_heads * self.out_features)

        return h_prime


# Graph Recurrent Attention Network for protein sequence generation
class GRAN(nn.Module):
    def __init__(self, node_features, hidden_dim, num_layers, n_heads, dropout=0.1, amino_acid_vocab_size=20):
        super(GRAN, self).__init__()

        self.node_features = node_features
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.n_heads = n_heads
        self.amino_acid_vocab_size = amino_acid_vocab_size  # Including special tokens if any

        # Graph attention layers
        self.gat_layers = nn.ModuleList()
        for _ in range(num_layers):
            self.gat_layers.append(GraphAttention(
                in_features=hidden_dim,
                out_features=hidden_dim // n_heads,
                n_heads=n_heads,
                dropout=dropout
            ))

        # Node feature embedding
        self.node_embedding = nn.Linear(node_features, hidden_dim)

        # RNN cell for sequence generation
        self.rnn_cell = nn.GRUCell(hidden_dim, hidden_dim)

        # Output projection to amino acid vocabulary
        self.output_projection = nn.Linear(hidden_dim, amino_acid_vocab_size)

        # Dropout
        self.dropout = nn.Dropout(dropout)

    def _process_graph(self, node_features, adjacency_matrix):
        """
        Process the graph with graph attention layers
        """
        batch_size, num_nodes = node_features.size(0), node_features.size(1)

        # Embed node features
        h = self.node_embedding(node_features)  # [batch_size, num_nodes, hidden_dim]

        # Apply graph attention layers
        for gat_layer in self.gat_layers:
            h = gat_layer(h, adjacency_matrix)
            h = F.elu(h)
            h = self.dropout(h)

        return h

    def forward(self, node_features, adjacency_matrix, target_sequences=None, max_length=100):
        """
        Forward pass for training or generation

        Args:
            node_features: Tensor [batch_size, num_nodes, node_features]
            adjacency_matrix: Tensor [batch_size, num_nodes, num_nodes]
            target_sequences: Tensor [batch_size, seq_length] (optional, for training)
            max_length: Maximum sequence length for generation

        Returns:
            If training (target_sequences is provided):
                logits: Tensor [batch_size, seq_length, amino_acid_vocab_size]
            If generating (target_sequences is None):
                generated_sequences: Tensor [batch_size, max_length]
        """
        batch_size, num_nodes = node_features.size(0), node_features.size(1)

        # Process the graph to get node embeddings
        node_embeddings = self._process_graph(node_features, adjacency_matrix)  # [batch_size, num_nodes, hidden_dim]

        # Aggregate node embeddings to get graph representation
        # Using mean aggregation for simplicity
        graph_embedding = torch.mean(node_embeddings, dim=1)  # [batch_size, hidden_dim]

        # Initialize hidden state with graph embedding
        h_t = graph_embedding  # [batch_size, hidden_dim]

        if target_sequences is not None:
            # Training mode
            seq_length = target_sequences.size(1)
            logits = torch.zeros(batch_size, seq_length, self.amino_acid_vocab_size).to(node_features.device)

            # Start token (could be a special token or just the first amino acid)
            x_t = torch.zeros(batch_size, self.hidden_dim).to(node_features.device)

            for t in range(seq_length):
                # Update hidden state
                h_t = self.rnn_cell(x_t, h_t)

                # Predict next amino acid
                output = self.output_projection(h_t)  # [batch_size, amino_acid_vocab_size]
                logits[:, t, :] = output

                # Teacher forcing: use target instead of prediction
                if t < seq_length - 1:
                    x_t = self.node_embedding(F.one_hot(target_sequences[:, t], num_classes=self.amino_acid_vocab_size).float())

            return logits

        else:
            # Generation mode
            generated_sequences = torch.zeros(batch_size, max_length, dtype=torch.long).to(node_features.device)

            # Start token (assuming 0 is the start token)
            x_t = torch.zeros(batch_size, self.hidden_dim).to(node_features.device)

            for t in range(max_length):
                # Update hidden state
                h_t = self.rnn_cell(x_t, h_t)

                # Predict next amino acid
                output = self.output_projection(h_t)  # [batch_size, amino_acid_vocab_size]
                prob = F.softmax(output, dim=-1)

                # Sample from distribution or take argmax
                next_aa = torch.multinomial(prob, 1).squeeze(-1)  # [batch_size]
                generated_sequences[:, t] = next_aa

                # Prepare input for next step
                x_t = self.node_embedding(F.one_hot(next_aa, num_classes=self.amino_acid_vocab_size).float())

            return generated_sequences


# Data loading and processing from the notebook
def load_protein_data(parent_folder, max_proteins=None, chunk_length=50):
    """
    Load protein data from folder structure similar to the notebook
    """
    random.seed(42)

    PROTEINS = Database()
    SUBPROTEINS = Database()

    folders = [name for name in os.listdir(parent_folder)
               if os.path.isdir(os.path.join(parent_folder, name))]

    print(f"Found {len(folders)} protein folders")

    # Load only max_proteins if specified
    if max_proteins:
        folders = folders[:max_proteins]

    # Get current working directory
    cwd = os.getcwd()

    for folder in folders:
        filename = f"{cwd}/{parent_folder}/{folder}/{folder}{folder[-2:]}_atoms.pkl"
        try:
            with open(filename, 'rb') as f:
                data = pkl.load(f)
                protein = Protein(data)
                PROTEINS.append(protein)
        except Exception as e:
            print(folder, 'corrupted or not found')

    PROTEINS.randomize

    # Create subsequences
    for protein in PROTEINS:
        aas, cas = protein.explode(chunk_length)
        for i in range(len(aas)):
            subprotein = Protein(aas=aas[i], cas=cas[i])
            SUBPROTEINS.append(subprotein)

    SUBPROTEINS.randomize

    # Limit size if needed
    if max_proteins:
        SUBPROTEINS = SUBPROTEINS[:max_proteins * 10]  # Arbitrary limit

    print(f"{SUBPROTEINS.len} subprotein sequences in database")

    return PROTEINS, SUBPROTEINS


def get_config_ranges(all_distances, n_ranges=6, mode='percentile'):
    """
    Define contact map configuration ranges
    """
    if mode == 'percentile':
        # even distribution of contacts within the ranges
        limits = np.percentile(all_distances, [100*i/n_ranges for i in range(n_ranges)] + [99.6]).tolist()
    elif mode == 'distance':
        # even distance ranges
        min_d, max_d = all_distances.min(), all_distances.max()
        std, mean = all_distances.std(), all_distances.mean()
        min_std, max_std = mean-2*std, mean+2*std
        limits = [min_std + i*((max_std-min_std)/n_ranges) for i in range(n_ranges)] + [max_d]
        limits[0] = min_d
    else:
        print("Define config mode, mode in ['percentile', 'distances']")
        return False

    print('Limits', limits)

    CONFIGS = []
    for i in range(len(limits)-1):
        CONFIGS.append({'lower': limits[i], 'upper': limits[i+1]})

    return CONFIGS


def prepare_data_for_training(SUBPROTEINS, CONFIGS, UNIQUE_AA, chunk_length):
    """
    Prepare data for GRAN model training
    """
    # Add contact maps to proteins
    for protein in SUBPROTEINS:
        for config in CONFIGS:
            protein.add_contact_map(config['lower'], config['upper'], True)

    # Prepare amino acid sequences (these will be our targets)
    aa_sequences = []
    for protein in SUBPROTEINS:
        # Convert amino acid sequence to indices
        aa_indices = [list(UNIQUE_AA).index(aa) for aa in protein.aa]
        aa_sequences.append(aa_indices)

    # Prepare contact maps (will be our graph adjacency matrices)
    contact_maps = []
    for protein in SUBPROTEINS:
        # Stack all contact maps for each protein
        contact_map = np.stack(protein.contact_maps, axis=-1)
        contact_maps.append(contact_map)

    # Convert to tensors
    aa_sequences_tensor = [torch.tensor(seq, dtype=torch.long) for seq in aa_sequences]
    contact_maps_tensor = [torch.tensor(cmap, dtype=torch.float32) for cmap in contact_maps]

    # Create node features (for simplicity, we'll use one-hot encodings of amino acids)
    node_features = []
    for protein in SUBPROTEINS:
        # Get one-hot encoding
        oh = protein.get_oh(list(UNIQUE_AA))
        node_features.append(oh)

    node_features_tensor = [torch.tensor(nf, dtype=torch.float32) for nf in node_features]

    return aa_sequences_tensor, contact_maps_tensor, node_features_tensor


def collate_batch(batch):
    """
    Custom collate function for padding sequences of variable length
    """
    aa_seqs, contact_maps, node_feats = zip(*batch)

    # Pad sequences
    padded_aa_seqs = torch.nn.utils.rnn.pad_sequence(aa_seqs, batch_first=True)

    # Pad contact maps and node features
    max_len = max([cm.size(0) for cm in contact_maps])
    padded_contact_maps = []
    padded_node_feats = []

    for i in range(len(contact_maps)):
        cm = contact_maps[i]
        nf = node_feats[i]

        pad_size = max_len - cm.size(0)
        if pad_size > 0:
            # Pad contact map
            padded_cm = F.pad(cm, (0, 0, 0, pad_size, 0, pad_size), "constant", 0)
            padded_contact_maps.append(padded_cm)

            # Pad node features
            padded_nf = F.pad(nf, (0, 0, 0, pad_size), "constant", 0)
            padded_node_feats.append(padded_nf)
        else:
            padded_contact_maps.append(cm)
            padded_node_feats.append(nf)

    padded_contact_maps = torch.stack(padded_contact_maps)
    padded_node_feats = torch.stack(padded_node_feats)

    return padded_aa_seqs, padded_contact_maps, padded_node_feats


def create_dataloader(aa_sequences, contact_maps, node_features, batch_size=32, shuffle=True):
    """
    Create a dataloader from the prepared data
    """
    dataset = list(zip(aa_sequences, contact_maps, node_features))
    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collate_batch
    )
    return dataloader


def train_model(model, train_loader, val_loader, num_epochs=10, lr=1e-4, device='cpu'):
    """
    Train the GRAN model
    """
    # Loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Training history
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        # Training
        model.train()
        epoch_train_loss = 0

        for i, (aa_seqs, contact_maps, node_feats) in enumerate(train_loader):
            # Move data to device
            aa_seqs = aa_seqs.to(device)
            contact_maps = contact_maps.to(device)
            node_feats = node_feats.to(device)

            # Select the first contact map (for simplicity)
            # In a more complex model, you could combine multiple contact maps
            adjacency_matrix = contact_maps[:, :, :, 0]

            # Forward pass
            logits = model(node_feats, adjacency_matrix, aa_seqs)

            # Reshape for loss calculation
            batch_size, seq_len, vocab_size = logits.size()
            logits_flat = logits.view(-1, vocab_size)
            targets_flat = aa_seqs.view(-1)

            # Calculate loss
            loss = criterion(logits_flat, targets_flat)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_train_loss += loss.item()

            if i % 10 == 0:
                print(f"Epoch {epoch+1}/{num_epochs}, Batch {i}/{len(train_loader)}, Loss: {loss.item():.4f}")

        avg_train_loss = epoch_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation
        model.eval()
        epoch_val_loss = 0

        with torch.no_grad():
            for aa_seqs, contact_maps, node_feats in val_loader:
                # Move data to device
                aa_seqs = aa_seqs.to(device)
                contact_maps = contact_maps.to(device)
                node_feats = node_feats.to(device)

                # Select the first contact map
                adjacency_matrix = contact_maps[:, :, :, 0]

                # Forward pass
                logits = model(node_feats, adjacency_matrix, aa_seqs)

                # Reshape for loss calculation
                batch_size, seq_len, vocab_size = logits.size()
                logits_flat = logits.view(-1, vocab_size)
                targets_flat = aa_seqs.view(-1)

                # Calculate loss
                loss = criterion(logits_flat, targets_flat)
                epoch_val_loss += loss.item()

        avg_val_loss = epoch_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

    return train_losses, val_losses


def generate_protein_sequence(model, contact_map, node_features, device, max_length=100):
    """
    Generate a protein sequence using the trained GRAN model
    """
    # Prepare inputs
    contact_map = contact_map.unsqueeze(0).to(device)  # [1, num_nodes, num_nodes]
    node_features = node_features.unsqueeze(0).to(device)  # [1, num_nodes, node_features]

    # Set model to evaluation mode
    model.eval()

    with torch.no_grad():
        # Generate sequence
        generated_ids = model(node_features, contact_map, max_length=max_length)

    # Convert ids to amino acids
    amino_acids = list(UNIQUE_AA)  # Use the same amino acid mapping as during training
    protein_sequence = ""

    for aa_id in generated_ids[0]:
        if aa_id.item() < len(amino_acids):
            protein_sequence += amino_acids[aa_id.item()]

    return protein_sequence


# Main execution
def main():
    # Parameters
    parent_folder = "nanos_networkx_small"  # Update this to your data path
    chunk_length = 50
    max_proteins = 100  # Limit number of proteins for faster execution
    n_ranges = 6  # Number of contact map ranges

    # Set device
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Load protein data
    PROTEINS, SUBPROTEINS = load_protein_data(parent_folder, max_proteins, chunk_length)

    # Get unique amino acids
    UNIQUE_AA = set()
    for protein in PROTEINS:
        UNIQUE_AA.update(protein.aa)
    UNIQUE_AA.update(['NUL'])  # Add padding token
    UNIQUE_AA = sorted(list(UNIQUE_AA))
    print(f"Unique amino acids: {len(UNIQUE_AA)}")

    # Get distance statistics
    all_distances = []
    for protein in SUBPROTEINS:
        # Flatten the distance matrix and remove zeros
        flat_distances = protein.D.flatten()
        flat_distances = flat_distances[flat_distances != 0.]
        all_distances.extend(flat_distances)
    all_distances = np.array(all_distances)

    # Define contact map configurations
    CONFIGS = get_config_ranges(all_distances, n_ranges, mode='percentile')

    # Prepare data for training
    aa_sequences, contact_maps, node_features = prepare_data_for_training(
        SUBPROTEINS, CONFIGS, UNIQUE_AA, chunk_length
    )

    # Split into train and validation sets
    split_idx = int(0.8 * len(aa_sequences))
    train_aa = aa_sequences[:split_idx]
    train_cm = contact_maps[:split_idx]
    train_nf = node_features[:split_idx]

    val_aa = aa_sequences[split_idx:]
    val_cm = contact_maps[split_idx:]
    val_nf = node_features[split_idx:]

    # Create dataloaders
    train_loader = create_dataloader(train_aa, train_cm, train_nf, batch_size=32)
    val_loader = create_dataloader(val_aa, val_cm, val_nf, batch_size=32)

    # Model parameters
    node_features_dim = node_features[0].size(1)  # One-hot encoding size
    hidden_dim = 64
    num_layers = 2
    n_heads = 4
    amino_acid_vocab_size = len(UNIQUE_AA)

    # Create model
    model = GRAN(
        node_features=node_features_dim,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        n_heads=n_heads,
        dropout=0.1,
        amino_acid_vocab_size=amino_acid_vocab_size
    ).to(device)

    # Train model
    train_losses, val_losses = train_model(
        model, train_loader, val_loader,
        num_epochs=10, lr=1e-4, device=device
    )

    # Plot training progress
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Progress')
    plt.legend()
    plt.show()

    # Generate a sample protein sequence
    sample_protein = SUBPROTEINS[0]
    sample_contact_map = torch.tensor(sample_protein.contact_maps[0], dtype=torch.float32)
    sample_node_features = torch.tensor(sample_protein.get_oh(list(UNIQUE_AA)), dtype=torch.float32)

    generated_sequence = generate_protein_sequence(
        model, sample_contact_map, sample_node_features, device, max_length=50
    )

    print("Original sequence:", ''.join(sample_protein.aa))
    print("Generated sequence:", generated_sequence)

    # Save model
    torch.save(model.state_dict(), "gran_protein_model.pt")
    print("Model saved to gran_protein_model.pt")


if __name__ == "__main__":
    main()

Using device: mps
Found 3024 protein folders
1000 subprotein sequences in database
Unique amino acids: 22
Limits [2.7116676295381366, 8.907124735817874, 12.803363491478237, 16.307353276053085, 20.03517440197068, 24.9237173445081, 39.793287712983116]
